# Fine-Tune XTTS On Shona In Colab

This notebook is designed for free Colab GPU use.

It supports two repo-loading paths:
- clone the repo from GitHub
- upload `cloner.zip` directly from your machine into Colab


## Outline

1. Check GPU runtime
2. Optionally mount Drive for checkpoints
3. Load the repo into `/content/cloner`
4. Install dependencies
5. Build the filtered Shona dataset from Hugging Face
6. Start XTTS fine-tuning
7. Resume from checkpoints later


In [1]:
# Step 1: verify that Colab gave you a GPU runtime.
!nvidia-smi || true

import torch
HAS_CUDA = torch.cuda.is_available()
print('cuda:', HAS_CUDA)
if HAS_CUDA:
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No CUDA GPU runtime is attached.')
    print('In Colab use Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')


/bin/bash: line 1: nvidia-smi: command not found
cuda: False
No CUDA GPU runtime is attached.
In Colab use Runtime > Change runtime type > GPU, reconnect, then rerun this cell.


In [2]:
# Step 2: mount Drive only if you want checkpoints to persist automatically.
USE_DRIVE = False

PROJECT_ROOT = '/content/cloner'
CHECKPOINT_DIR = '/content/checkpoints/shona_xtts'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/shona_xtts'
    CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/shona_xtts'

print('CHECKPOINT_DIR =', CHECKPOINT_DIR)


CHECKPOINT_DIR = /content/checkpoints/shona_xtts


## Step 3A - Clone From GitHub

Use this if your repo is on GitHub. Otherwise skip to Step 3B.


In [3]:
# Step 3A: clone from GitHub.
# Replace the URL before running.
GIT_REPO_URL = ''

import os, shutil
if GIT_REPO_URL:
    if os.path.exists(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)
    !git clone "$GIT_REPO_URL" "$PROJECT_ROOT"
    !ls -la "$PROJECT_ROOT"
else:
    print('Set GIT_REPO_URL or skip this cell and use Step 3B.')


Set GIT_REPO_URL or skip this cell and use Step 3B.


## Step 3B - Upload `cloner.zip` Directly

Use this if the repo is not on GitHub.

Before running this cell, create a zip of the local repo named `cloner.zip`.


In [4]:
# Step 3B: extract cloner.zip already present in /content to /content/cloner.
import os, zipfile, shutil
assert os.path.exists('/content/cloner.zip'), 'cloner.zip not found in /content'
if os.path.exists(PROJECT_ROOT):
    shutil.rmtree(PROJECT_ROOT)

with zipfile.ZipFile('/content/cloner.zip', 'r') as zf:
    zf.extractall('/content')

assert os.path.exists(PROJECT_ROOT), 'Expected /content/cloner after extracting cloner.zip'
print('Repo extracted to', PROJECT_ROOT)
!ls -la "$PROJECT_ROOT" | sed -n '1,20p'


AssertionError: cloner.zip not found in /content

In [ ]:
# Step 3C: verify the repo is present before continuing.
import os
assert os.path.exists(PROJECT_ROOT), 'Repo missing. Run Step 3A or Step 3B first.'
assert os.path.exists(f'{PROJECT_ROOT}/pyproject.toml'), 'pyproject.toml missing. The repo did not extract correctly.'
print('Repo is ready at', PROJECT_ROOT)


In [ ]:
# Step 4: install dependencies.
import os
assert os.path.exists(f'{PROJECT_ROOT}/pyproject.toml'), 'Repo missing. Run Step 3A or 3B first.'
%cd /content/cloner
!python -V
!pip install -U pip setuptools wheel
!pip install uv
!uv sync


In [ ]:
# Step 5: build the filtered Shona dataset directly from Hugging Face.
import os, shutil
assert os.path.exists(f'{PROJECT_ROOT}/scripts/build_hf_xtts_dataset.py'), 'Missing build_hf_xtts_dataset.py. Repo not loaded correctly.'

TARGET_DATASET_DIR = '/content/cloner/data/sna_xtts_ft_filtered'
if os.path.exists(TARGET_DATASET_DIR):
    shutil.rmtree(TARGET_DATASET_DIR)

%cd /content/cloner
!uv run python scripts/build_hf_xtts_dataset.py \
  --output ./data/sna_xtts_ft_filtered \
  --min-quality 70 \
  --min-duration 3 \
  --max-duration 20 \
  --max-clips-per-speaker 200

!find /content/cloner/data/sna_xtts_ft_filtered/wavs -type f -name '*.wav' | wc -l
!du -sh /content/cloner/data/sna_xtts_ft_filtered


In [ ]:
# Step 6: launch XTTS fine-tuning.
# The dataset is Shona. `--xtts-language en` is only the XTTS-supported language token workaround.
if not HAS_CUDA:
    raise RuntimeError('Stop here until Colab gives you a GPU runtime. Rerun Step 1 after changing runtime type.')

%cd /content/cloner
!uv run python scripts/finetune_xtts.py \
  --dataset ./data/sna_xtts_ft_filtered \
  --output "$CHECKPOINT_DIR" \
  --xtts-language en \
  --batch-size 1 \
  --grad-accum 84 \
  --steps 5000 \
  --save-step 500 \
  --workers 2


In [ ]:
# Step 7: resume later if Colab disconnects.
LATEST_CHECKPOINT = ''

if LATEST_CHECKPOINT:
    %cd /content/cloner
    !uv run python scripts/finetune_xtts.py \
      --dataset ./data/sna_xtts_ft_filtered \
      --output "$CHECKPOINT_DIR" \
      --xtts-language en \
      --resume "$LATEST_CHECKPOINT" \
      --batch-size 1 \
      --grad-accum 84 \
      --steps 5000 \
      --save-step 500 \
      --workers 2
else:
    print('Set LATEST_CHECKPOINT to resume from an interrupted run.')


## Notes

- If Google Drive auth is flaky, keep `USE_DRIVE = False` and train under `/content`.
- If the Colab session resets, rerun all cells after Step 1.
- XTTS still uses `en` as the supported internal language token even though the dataset is Shona.
